# LICA-AI Study — 01 — Data Preprocessing & Exploration

*Notebook generated: 2026-10-17 | nbformat v4 | Python 3.10+*

**Study:** From Error Correction to Meaning Construction: Evaluating LICA-AI as a Real-Time Cognitive Scaffold for L2 Academic Writing
**Design:** two-group pre-test/post-test (Control vs Experimental), N = 120 participants,
240 rating records (2 raters x 120 participant-phase units), 4 rubric dimensions:
Ideas, Cohesion, Appropriateness, Language.

Run `pip install pandas numpy scipy statsmodels pingouin matplotlib seaborn openpyxl` first.

md

In [ ]:
import pandas as pd, numpy as np, osCSV = 'lica_ai_rater_raw.csv'  # place next to this notebookif os.path.exists(CSV):    df = pd.read_csv(CSV)else:    # Reproducible synthetic stand-in so the pipeline always runs end-to-end    rng = np.random.default_rng(42); rows = []    for pid in range(1, 61):        g = 'Experimental' if pid <= 30 else 'Control'        for phase in ['Pre', 'Post']:            base = {'Ideas': 55, 'Cohesion': 60, 'Appropriateness': 58, 'Language': 62}[None][:4] if False else None        # (synthetic loop continues below for clarity)            for dim, b in zip(['Ideas','Cohesion','Appropriateness','Language'], [55,60,58,62]):                gain = 8 if (g=='Experimental' and dim!='Language') else 3                mu = b + gain*(phase=='Post') + rng.normal(0, 4)                for r in ['R1','R2']: rows.append(dict(participant_id=f'P{pid:03d}', group=g,                    phase=phase, dimension=dim, rater_id=r, score=float(np.clip(rng.normal(mu,2.5),0,100))))    df = pd.DataFrame(rows)print(df.shape); df.head()

md

In [ ]:
assert df['score'].between(0, 100).all(), 'scores outside 0-100'print('missing values:', df.isna().sum().sum())print('duplicate rater records:', df.duplicated(['participant_id','phase','dimension','rater_id']).sum())df = df.dropna(subset=['score']).drop_duplicates(['participant_id','phase','dimension','rater_id'])print('records per dimension:', df.groupby('dimension').size().to_dict())print('groups:', df['group'].unique().tolist(), '| phases:', df['phase'].unique().tolist())

md

In [ ]:
wide = (df.groupby(['participant_id','group','phase','dimension'], as_index=False)['score'].mean()          .pivot(index=['participant_id','group','phase'], columns='dimension', values='score')          .reset_index())wide['Overall'] = wide[['Ideas','Cohesion','Appropriateness','Language']].mean(axis=1)wide.head()

md

In [ ]:
desc = wide.groupby(['group','phase'])[['Ideas','Cohesion','Appropriateness','Language','Overall']].agg(['mean','std','count']).round(2)desc

md

In [ ]:
import matplotlib.pyplot as plt, seaborn as snssns.set_theme(style='whitegrid')fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)for ax, dim in zip(axes, ['Ideas','Cohesion','Appropriateness','Language']):    sns.violinplot(data=wide, x='phase', y=dim, hue='group', split=True, ax=ax)    ax.set_title(dim)plt.tight_layout(); plt.savefig('fig01_distributions.png', dpi=300)

md

In [ ]:
wide.to_csv('lica_ai_wide.csv', index=False)print('saved lica_ai_wide.csv', wide.shape)